# Summary Buffer Memory

> **A hybrid memory that keeps recent messages word-for-word while summarizing older history.**

Think about how you remember a long phone call with a friend. The last few sentences? You recall them almost word-for-word. That's your short-term memory. The earlier parts? You remember the gist, not the exact words. That's your long-term memory.

Summary Buffer Memory works the same way. It keeps a **buffer zone** of the most recent messages in their original form. Everything older gets compressed into a **running summary**. The LLM receives both pieces on every call: the summary for historical context, and the raw recent messages for precision.

Pure buffer memory (storing every message) gives exact recall but costs more tokens with each turn. Pure summary memory (compressing everything) saves tokens but loses recent detail. Summary Buffer Memory combines both strengths. It's especially valuable for customer support bots, coaching agents, and project management assistants. These agents need to remember the full arc of a conversation while responding precisely to the latest message.

The core engineering challenge is the **transition threshold**: the point where messages age out of the buffer and fold into the summary. Set it too high and you waste tokens. Set it too low and you lose important detail.

**By the end of this notebook you'll understand:**
- How to build a summary buffer memory from scratch with the OpenAI SDK.
- How incremental summarization works (updating a summary with newly evicted messages).
- How to tune the buffer size and observe its effect on token usage.
- When this technique wins and when it quietly fails.

## Key Concepts

- **Buffer zone**: The most recent *k* messages stored word-for-word. These preserve exact wording, nuance, and detail for immediate reasoning.
- **Running summary**: A short paragraph that captures the key facts and themes from all messages that have aged out of the buffer. The LLM generates this summary.
- **Context window**: The maximum number of tokens (word-pieces the model uses internally) a model can read in one call. For example, GPT-4o supports up to 128k tokens.
- **Transition threshold**: The trigger point at which messages move from the buffer into the summary. You can base it on message count or token count.
- **Incremental summarization**: Instead of re-summarizing the full history each time, we take the existing summary and the newly evicted messages, then produce an updated summary in a single LLM call. This keeps summarization cost low.
- **Token budget allocation**: The split of available context-window tokens between the summary and the buffer. A typical allocation reserves 20-30% for the summary and 70-80% for recent messages.
- **Prompt assembly**: The process of stitching together `[system prompt] + [summary] + [recent buffer messages]` into one prompt for the LLM.

## Architecture

<p align="center">
  <img src="../../images/diagrams/04_summary_buffer_memory.svg" alt="Summary Buffer Memory architecture diagram" width="720"/>
</p>

<details><summary>Mermaid source</summary>

```mermaid
flowchart LR
    User["User Message"] --> Buffer["Buffer\n(recent K messages)"]
    Buffer --> ThresholdCheck{"Buffer exceeds\nthreshold?"}
    ThresholdCheck -- No --> PromptAssembly["Prompt Assembly"]
    ThresholdCheck -- Yes --> Evict["Evict oldest\nmessages"]
    Evict --> Summarizer["Summarizer\n(LLM call)"]
    Summarizer --> SummaryStore["Summary Store\n(running summary)"]
    SummaryStore --> PromptAssembly
    Buffer --> PromptAssembly
    PromptAssembly --> LLM["LLM\n[summary] + [buffer]"]
    LLM --> Response["Response"]
    Response --> Buffer
```

</details>

Here's how the data flows. New user messages enter the buffer. When the buffer exceeds its configured threshold, the oldest messages get evicted. Those evicted messages feed a summarizer LLM call that updates the running summary. The final prompt concatenates the system prompt, the summary, and the recent buffer messages. The LLM responds, and that response goes back into the buffer. The cycle repeats on every turn.

## Setup

Install dependencies and configure API access. You'll need an `OPENAI_API_KEY` environment variable set in a `.env` file.

In [ ]:
%pip install -q openai python-dotenv tiktoken

Import the OpenAI SDK, `tiktoken` (OpenAI's tokenizer library for counting tokens), and standard library helpers.

In [ ]:
import os
import json
import copy
from dotenv import load_dotenv
import tiktoken
from openai import OpenAI

load_dotenv()  # reads OPENAI_API_KEY from .env

assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in your .env file"

client = OpenAI()

# We'll use this encoder throughout to count tokens.
# "cl100k_base" is the tokenizer used by GPT-4o and GPT-4.
encoder = tiktoken.get_encoding("cl100k_base")

## Implementation

We'll build a `SummaryBufferMemory` class that:
1. Stores recent messages in a buffer (a plain Python list).
2. Maintains a running summary of older history.
3. Evicts the oldest messages when the buffer exceeds a token threshold.
4. Uses an LLM call to fold evicted messages into the summary.
5. Assembles the final prompt as `[system] + [summary] + [buffer]`.

### Token counting helper

We need a way to count how many tokens a list of messages uses. This tells us when the buffer has grown too large.

In [ ]:
def count_message_tokens(messages: list[dict], enc: tiktoken.Encoding = encoder) -> int:
    """Count the total tokens in a list of chat messages.

    Each message has overhead tokens for role formatting.
    We add 4 tokens per message as an approximation of that overhead.
    """
    total = 0
    for msg in messages:
        total += 4  # role + formatting overhead
        total += len(enc.encode(msg["content"]))
    return total

### The summarizer

This function takes the current running summary and a batch of newly evicted messages. It asks the LLM to produce an updated summary that incorporates the new information.

Think of it like updating meeting notes. You have yesterday's notes (the existing summary) and today's discussion points (the evicted messages). You merge them into one concise document.

In [ ]:
SUMMARY_SYSTEM_PROMPT = """You are a conversation summarizer. Your job is to maintain
a running summary of a conversation. You will receive:
1. The current summary (may be empty if this is the start).
2. New messages that need to be incorporated.

Produce an updated summary that:
- Preserves all key facts, names, preferences, and decisions.
- Stays concise (aim for 3-6 sentences).
- Uses third person ("The user said...", "The assistant suggested...").
- Drops filler and small talk while keeping substance.
"""


def update_summary(
    current_summary: str,
    evicted_messages: list[dict],
    model: str = "gpt-4o-mini",
) -> str:
    """Fold evicted messages into the running summary via an LLM call."""
    # Format the evicted messages as readable text
    evicted_text = "\n".join(
        f"{msg['role'].upper()}: {msg['content']}" for msg in evicted_messages
    )

    user_prompt = (
        f"Current summary:\n{current_summary if current_summary else '(empty)'}\n\n"
        f"New messages to incorporate:\n{evicted_text}\n\n"
        f"Write the updated summary:"
    )

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SUMMARY_SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        max_tokens=512,
        temperature=0.3,  # low temperature for consistent summaries
    )

    return response.choices[0].message.content

### The SummaryBufferMemory class

This is the core of the technique. The class manages two memory regions:
- **Buffer**: a list of recent messages (word-for-word).
- **Summary**: a string that captures older history.

When you call `chat()`, the class checks whether the buffer exceeds `max_buffer_tokens`. If it does, the oldest messages get evicted and folded into the summary. Then it assembles the full prompt and calls the LLM.

In [ ]:
class SummaryBufferMemory:
    """Hybrid memory: running summary + recent message buffer."""

    def __init__(
        self,
        model: str = "gpt-4o-mini",
        system_prompt: str | None = None,
        max_buffer_tokens: int = 300,
        max_tokens: int = 1024,
    ):
        self.model = model
        self.system_prompt = system_prompt
        self.max_buffer_tokens = max_buffer_tokens
        self.max_tokens = max_tokens

        # The two memory regions
        self.buffer: list[dict] = []      # recent messages (verbatim)
        self.summary: str = ""             # compressed older history

        # Bookkeeping
        self.turn_count: int = 0
        self.eviction_log: list[dict] = []  # tracks each eviction event



Next we add the eviction and prompt-building logic. `_evict_if_needed` pops the oldest messages from the buffer one at a time, then calls `update_summary` to fold them into the running summary. `_build_messages` stitches together the system prompt, the summary, and the remaining buffer into one prompt for the API.

In [ ]:
    def _evict_if_needed(self) -> None:
        """Move oldest messages from buffer to summary when over budget."""
        evicted_batch = []

        while (
            len(self.buffer) > 2
            and count_message_tokens(self.buffer) > self.max_buffer_tokens
        ):
            # Evict the oldest message (FIFO: first in, first out)
            evicted_batch.append(self.buffer.pop(0))

        if evicted_batch:
            old_summary = self.summary
            self.summary = update_summary(self.summary, evicted_batch, self.model)

            self.eviction_log.append({
                "turn": self.turn_count,
                "evicted_count": len(evicted_batch),
                "summary_before": old_summary,
                "summary_after": self.summary,
            })

    def _build_messages(self) -> list[dict]:
        """Assemble the full message list for the API call."""
        messages = []

        # Inject the summary as a system-level context block
        if self.summary:
            summary_block = f"Summary of earlier conversation:\n{self.summary}"
            if self.system_prompt:
                messages.append({
                    "role": "system",
                    "content": f"{self.system_prompt}\n\n{summary_block}",
                })
            else:
                messages.append({"role": "system", "content": summary_block})
        elif self.system_prompt:
            messages.append({"role": "system", "content": self.system_prompt})

        # Append the recent buffer messages
        messages.extend(self.buffer)
        return messages



The `chat` method ties everything together. It appends the user message, triggers eviction if needed, builds the assembled prompt, and calls the LLM. The response goes back into the buffer for future context.

In [ ]:
    def chat(self, user_input: str) -> str:
        """Send a message and get a response."""
        self.buffer.append({"role": "user", "content": user_input})
        self.turn_count += 1

        # Check if we need to evict before calling the LLM
        self._evict_if_needed()

        # Build the prompt and call the API
        messages = self._build_messages()

        response = client.chat.completions.create(
            model=self.model,
            messages=messages,
            max_tokens=self.max_tokens,
        )

        assistant_text = response.choices[0].message.content
        self.buffer.append({"role": "assistant", "content": assistant_text})

        return assistant_text



Finally, we add utility methods for inspecting and resetting the memory. `get_buffer` and `get_summary` let you peek at each region. `get_buffer_token_count` tells you how close the buffer is to the eviction threshold.

In [ ]:
    def get_buffer(self) -> list[dict]:
        """Return a copy of the current buffer."""
        return copy.deepcopy(self.buffer)

    def get_summary(self) -> str:
        """Return the current running summary."""
        return self.summary

    def get_buffer_token_count(self) -> int:
        """Return the current token count of the buffer."""
        return count_message_tokens(self.buffer)

    def clear(self) -> None:
        """Reset both memory regions."""
        self.buffer.clear()
        self.summary = ""
        self.turn_count = 0
        self.eviction_log.clear()

    def __repr__(self) -> str:
        return (
            f"SummaryBufferMemory("
            f"buffer={len(self.buffer)} msgs, "
            f"summary={'yes' if self.summary else 'empty'}, "
            f"turns={self.turn_count})"
        )

## Example Run

Let's run a multi-turn conversation and watch the summary buffer in action. We'll use a small buffer threshold (300 tokens) so that evictions happen quickly. In production you'd set this higher.

Create the memory and run a conversation. We'll print the agent's reply after each turn.

In [ ]:
memory = SummaryBufferMemory(
    system_prompt="You are a helpful, concise assistant. Keep replies under 2 sentences.",
    max_buffer_tokens=300,
)

exchanges = [
    "Hi! My name is Alice and I'm a machine-learning engineer in Berlin.",
    "I'm building a chatbot for our customer support team. We get about 500 tickets a day.",
    "The main issue is that our current bot forgets context after 3 messages.",
    "We use Python and FastAPI on the backend. The bot runs on GPT-4o.",
    "Our customers mostly ask about billing, shipping, and returns.",
    "I also want the bot to remember user preferences across sessions.",
    "We store conversation logs in PostgreSQL right now.",
    "What memory technique would you recommend for our use case?",
    "By the way, what's my name and where do I work?",
]

for msg in exchanges:
    print(f"\n{'='*60}")
    print(f"Turn {memory.turn_count + 1}")
    print(f"{'='*60}")
    print(f"User:  {msg}")
    reply = memory.chat(msg)
    print(f"Agent: {reply}")
    print(f"\n  Buffer: {len(memory.buffer)} messages ({memory.get_buffer_token_count()} tokens)")
    print(f"  Summary: {'[empty]' if not memory.summary else memory.summary[:100] + '...'}")

### Inspecting the memory state

After 9 turns, some messages have been evicted and summarized. Let's look at what's in each region.

In [ ]:
print("RUNNING SUMMARY")
print("-" * 40)
print(memory.get_summary() or "(empty)")

print("\nBUFFER (recent messages)")
print("-" * 40)
for i, msg in enumerate(memory.get_buffer()):
    role_label = "USER" if msg["role"] == "user" else "ASST"
    preview = msg["content"][:90] + ("..." if len(msg["content"]) > 90 else "")
    print(f"  [{i}] {role_label}: {preview}")

print(f"\nBuffer token count: {memory.get_buffer_token_count()}")
print(f"Buffer threshold:   {memory.max_buffer_tokens}")

### Eviction log

Each time messages aged out of the buffer, the system recorded the event. This log shows how the summary evolved over time.

In [ ]:
print(f"Total eviction events: {len(memory.eviction_log)}\n")

for i, event in enumerate(memory.eviction_log):
    print(f"Eviction {i + 1} (after turn {event['turn']})")
    print(f"  Messages evicted: {event['evicted_count']}")
    print(f"  Summary before: {event['summary_before'][:80] or '(empty)'}...")
    print(f"  Summary after:  {event['summary_after'][:80]}...")
    print()

### What the LLM actually sees

Let's peek at the assembled prompt. This is exactly what gets sent to the API on the next call. Notice how the summary sits in the system message and the buffer messages follow.

In [ ]:
assembled = memory._build_messages()

print(f"Total messages in assembled prompt: {len(assembled)}")
print(f"Total tokens in assembled prompt:   {count_message_tokens(assembled)}\n")

for i, msg in enumerate(assembled):
    role = msg["role"].upper()
    content = msg["content"]
    if len(content) > 150:
        content = content[:150] + "..."
    print(f"[{i}] {role}:")
    print(f"    {content}")
    print()

## Comparison: Buffer vs. Summary Buffer

How does our hybrid approach compare to plain buffer memory in terms of token usage? Let's run the same conversation through both and measure.

In [ ]:
# Simulate plain buffer memory: count tokens as if we sent ALL messages each turn.
# We reuse the same exchanges list from above.

all_messages = []  # accumulates like a plain buffer
buffer_tokens_per_turn = []
summary_buffer_tokens_per_turn = []

# Reset summary buffer memory for a clean run
sbm = SummaryBufferMemory(
    system_prompt="You are a helpful, concise assistant. Keep replies under 2 sentences.",
    max_buffer_tokens=300,
)

for msg in exchanges:
    # Plain buffer: accumulate all messages
    all_messages.append({"role": "user", "content": msg})
    buffer_tokens_per_turn.append(count_message_tokens(all_messages))
    # Add a placeholder assistant reply to keep the count realistic
    all_messages.append({"role": "assistant", "content": "Acknowledged."})

    # Summary buffer: use the actual class
    sbm.chat(msg)
    assembled_msgs = sbm._build_messages()
    summary_buffer_tokens_per_turn.append(count_message_tokens(assembled_msgs))

print(f"{'Turn':<6} {'Buffer Memory':<18} {'Summary Buffer Memory'}")
print("-" * 48)
for i in range(len(exchanges)):
    print(f"{i+1:<6} {buffer_tokens_per_turn[i]:<18} {summary_buffer_tokens_per_turn[i]}")

print(f"\nFinal turn token usage:")
print(f"  Buffer memory:         {buffer_tokens_per_turn[-1]} tokens")
print(f"  Summary buffer memory: {summary_buffer_tokens_per_turn[-1]} tokens")
print(f"  Savings:               {buffer_tokens_per_turn[-1] - summary_buffer_tokens_per_turn[-1]} tokens")

## Tradeoffs

### When Summary Buffer Memory Works Well

- **Medium to long conversations** (20-100+ turns) where you need both historical context and precise recent detail. Customer support bots and coaching agents are common examples.
- **Token-budget-conscious applications**: the summary compresses older history, so cumulative token usage grows much slower than with pure buffer memory.
- **When recent context matters most**: the buffer zone keeps the last few exchanges in full fidelity. The model can reference exact words, numbers, and names from the recent turns.

### When It Breaks Down

- **Summary quality degrades over time.** Each incremental summarization call risks losing detail. After dozens of evictions, the summary may miss facts that mattered. There's no built-in way to verify that the summary is accurate.
- **Extra LLM calls add cost and latency.** Every eviction triggers a summarization call. For fast-paced conversations (sub-second response requirements), this overhead may be too high.
- **Harder to debug than plain buffer.** When the agent forgets something, you need to trace whether the information was lost during summarization or never reached the buffer. With plain buffer memory, the full history is always there.
- **Tuning is tricky.** The buffer threshold, the summarization prompt, and the model choice for summarization all interact. Getting the right balance takes experimentation.

## Further Reading

- [LangChain ConversationSummaryBufferMemory](https://python.langchain.com/docs/modules/memory/types/summary_buffer?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques): The framework implementation of this pattern with configurable token limits and LLM-based summarization.
- [MemGPT / Letta: Tiered Memory Management (arXiv:2310.08560)](https://arxiv.org/abs/2310.08560): Packer et al. introduce a multi-tiered memory architecture where the LLM manages its own context window, moving data between main context and external storage.
- [LlamaIndex ChatSummaryMemoryBuffer](https://docs.llamaindex.ai/en/stable/api_reference/memory/?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques): LlamaIndex's implementation that integrates summary-buffer memory with retrieval-augmented generation pipelines.
- [OpenAI Cookbook: Counting Tokens with tiktoken](https://cookbook.openai.com/examples/how_to_count_tokens_with_tiktoken?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques): Practical strategies for counting and managing tokens in chat applications.

---

*← Previous: [03: Summary Memory](../03_summary_memory/) · Next: [05: Token Buffer Memory](../05_token_buffer_memory/) →*

## 🧪 Try It Yourself

Three small challenges to deepen your understanding. Each should take 10-30 minutes.

### Challenge 1: Tune the eviction threshold
Modify the token threshold in `SummaryBufferMemory` to 500, 1000, and 2000 tokens. For each setting, run a 15-turn conversation and note how often `_evict_if_needed()` triggers. Record the final summary length each time.

### Challenge 2: Compression ratio tracking
After each eviction cycle, compute the ratio of tokens saved (evicted raw tokens minus summary tokens). Plot this compression ratio across turns. Identify the point where summarization gives diminishing returns.

### Challenge 3: Entity-aware summarization
Before calling `update_summary()`, extract entities from the evicted messages using the approach from 07 Entity Memory. Pass the entity list to the summarizer prompt so it preserves named entities. Compare recall on entity-specific questions before and after this change.


![](https://europe-west1-amt-views-tracker.cloudfunctions.net/amt-tracker?notebook=all-techniques--04-summary-buffer-memory--summary-buffer-memory)
